# 2026 Opponent Trends & Match Scouting

Use recent Wyscout **full Match Report PDFs** to study an upcoming opponent without using matches played on or after the scouting date. The notebook produces a reconciled match history, rolling form charts, a recent-form summary, and a shareable evidence report. Excel/CSV team-stat exports remain supported when available.

> Coaching boundary: the output describes evidence and trends. Analysts and coaches remain responsible for tactical interpretation, video confirmation, and final recommendations.

## 1. Setup
Enter the opponent name exactly as it appears in Wyscout. `SCOUTING_DATE` is the upcoming match date; that match and all later matches are automatically excluded. For useful trends, provide 5–10 earlier full Match Report PDFs.

In [ ]:
OPPONENT_NAME = 'USC Upstate Spartans'  # Exact Wyscout team name
SCOUTING_DATE = '2026-09-12'            # Upcoming CofC match date
RECENT_WINDOW = 5

# Leave blank to upload files when prompted. To reuse files in shared Drive,
# use a glob such as '/content/drive/MyDrive/CofC_Soccer/scouting/2026/usc_upstate/*.pdf'.
DRIVE_INPUT_GLOB = ''

if not OPPONENT_NAME.strip():
    raise ValueError('Fill in OPPONENT_NAME before continuing.')
if not SCOUTING_DATE.strip():
    raise ValueError('Fill in SCOUTING_DATE before continuing.')

## 2. Load the tested scouting code

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_ROOT = Path('/content/cofc-soccer-analytics')
if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/anissawilliams/cofc-soccer-analytics.git',
        str(REPO_ROOT),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
if importlib.util.find_spec('pypdf') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf>=5.0.0'], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Scouting code ready:', REPO_ROOT)

## 3. Select Wyscout files
Upload Wyscout **full Match Report PDFs** like `UNCW Seahawks - Elon Phoenix 1-0.pdf`. The notebook finds the Team Stats page automatically. `.xlsx`, `.xls`, and equivalent `.csv` team-stat exports also work. Event XML, effective-time XML, and player-only reports serve different purposes and are intentionally rejected here.

In [ ]:
import glob

if DRIVE_INPUT_GLOB.strip():
    from google.colab import drive
    drive.mount('/content/drive')
    input_paths = [Path(path) for path in glob.glob(DRIVE_INPUT_GLOB)]
else:
    from google.colab import files
    upload_dir = Path('/content/opponent_wyscout')
    upload_dir.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    input_paths = []
    for name, contents in uploaded.items():
        path = upload_dir / Path(name).name
        path.write_bytes(contents)
        input_paths.append(path)

input_paths = sorted(
    path for path in input_paths if path.suffix.lower() in {'.pdf', '.xlsx', '.xls', '.csv'}
)
if not input_paths:
    raise RuntimeError('No supported Wyscout Match Report PDFs or team-stat workbooks were selected.')
print(f'Selected {len(input_paths)} file(s):')
for path in input_paths:
    print('-', path.name)

## 4. Reconcile and inspect
The validation requires exactly two team rows per match and confirms the requested opponent exists. Duplicates are removed using date, match, and team identity.

In [ ]:
import pandas as pd
from IPython.display import display

from pipeline.scouting.opponent_trends import (
    build_opponent_trends,
    load_opponent_history,
    summarize_recent_form,
)

history = load_opponent_history(input_paths, OPPONENT_NAME, SCOUTING_DATE)
trends = build_opponent_trends(history, OPPONENT_NAME)
summary = summarize_recent_form(trends, RECENT_WINDOW)

print(f'Validated {len(trends)} {OPPONENT_NAME} matches before {SCOUTING_DATE}.')
display(trends[[
    'date', 'opponent_team', 'result', 'goals', 'goals_against',
    'xg', 'xg_against', 'shots', 'shots_on_target', 'possession_pct',
]].tail(RECENT_WINDOW))
display(pd.DataFrame([summary['averages']]))
print('Recent record:', summary['record'])

## 5. Visualize recent trends

In [ ]:
import matplotlib.pyplot as plt

plot_data = trends.tail(RECENT_WINDOW).copy()
labels = plot_data['date'].dt.strftime('%m/%d') + '\n' + plot_data['opponent_team'].astype(str)
colors = plot_data['result'].map({'W': '#2e8b57', 'D': '#d99b22', 'L': '#b23a48'})
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
x = range(len(plot_data))

axes[0, 0].plot(x, plot_data['xg'], marker='o', label='xG for')
axes[0, 0].plot(x, plot_data['xg_against'], marker='o', label='xG against')
axes[0, 0].set_title('Shot quality')
axes[0, 0].set_xlabel('Match date and opponent')
axes[0, 0].set_ylabel('Expected goals (xG)')
axes[0, 0].legend()

axes[0, 1].plot(x, plot_data['shots'], marker='o', label='Shots')
axes[0, 1].plot(x, plot_data['shots_on_target'], marker='o', label='Shots on target')
axes[0, 1].set_title('Shot volume')
axes[0, 1].set_xlabel('Match date and opponent')
axes[0, 1].set_ylabel('Shots (count)')
axes[0, 1].legend()

axes[1, 0].bar(x, plot_data['possession_pct'], color=colors)
axes[1, 0].axhline(50, color='black', linewidth=0.8, linestyle='--')
axes[1, 0].set_title('Possession % (color = result)')
axes[1, 0].set_xlabel('Match date and opponent')
axes[1, 0].set_ylabel('Possession (%)')

axes[1, 1].plot(x, plot_data['recoveries'], marker='o', label='Recoveries')
axes[1, 1].plot(x, plot_data['duels_won'], marker='o', label='Duels won')
axes[1, 1].set_title('Defensive activity')
axes[1, 1].set_xlabel('Match date and opponent')
axes[1, 1].set_ylabel('Actions (count)')
axes[1, 1].legend()

for axis in axes.flat:
    axis.set_xticks(list(x), labels, rotation=35, ha='right')
    axis.grid(axis='y', alpha=0.2)
fig.suptitle(f'{OPPONENT_NAME}: form before {SCOUTING_DATE}', fontsize=16, fontweight='bold')
plt.tight_layout()
chart_path = Path('/content/opponent_trends.png')
plt.savefig(chart_path, dpi=160, bbox_inches='tight')
plt.show()

## 6. Export the evidence packet
The Markdown report deliberately leaves tactical conclusions for analyst/video review.

In [ ]:
import json
import shutil

safe_name = ''.join(char.lower() if char.isalnum() else '_' for char in OPPONENT_NAME).strip('_')
output_dir = Path('/content') / f'{SCOUTING_DATE}_{safe_name}_scouting'
output_dir.mkdir(parents=True, exist_ok=True)
trends.to_csv(output_dir / 'opponent_match_history.csv', index=False)
(output_dir / 'recent_form_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
shutil.copy2(chart_path, output_dir / chart_path.name)

record = summary['record']
averages = summary['averages']
report = f'''# {OPPONENT_NAME} — Evidence Brief

Scouting date: {SCOUTING_DATE}  
Evidence window: {summary['date_from']} to {summary['date_to']}  
Matches: {summary['matches_in_window']} of {summary['matches_available']} available  
Record: {record['W']}W-{record['D']}D-{record['L']}L

## Recent averages

- Goals: {averages.get('goals', float('nan')):.2f}
- Goals against: {averages.get('goals_against', float('nan')):.2f}
- xG: {averages.get('xg', float('nan')):.2f}
- xG against: {averages.get('xg_against', float('nan')):.2f}
- Shots: {averages.get('shots', float('nan')):.2f}
- Shots on target: {averages.get('shots_on_target', float('nan')):.2f}
- Possession: {averages.get('possession_pct', float('nan')):.1f}%

## Analyst and video review

- What persists across opponents and game states?
- What changes home versus away?
- Which numerical trend is supported by repeatable video evidence?
- What should players recognize, do, avoid, or exploit?

> These are descriptive trends from the supplied matches, not causal claims or final tactical recommendations.
'''
(output_dir / 'evidence_brief.md').write_text(report, encoding='utf-8')
archive = shutil.make_archive(str(output_dir), 'zip', output_dir)
print(report)
print('Evidence packet:', archive)

from google.colab import files
files.download(archive)